In [ ]:
!pip install -q hydra-core tensorboardX treelib matplotlib

In [ ]:
from pathlib import Path
from datetime import datetime
import os
import sys
import shutil
import csv
import pickle
import random
import logging
from pathlib import Path
from math import log
from itertools import combinations
import hydra
from omegaconf import OmegaConf
from tensorboardX import SummaryWriter
from omegaconf import DictConfig
import numpy as np
import treelib
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical, Distribution
from torch.utils.data.sampler import BatchSampler, SubsetRandomSampler
from typing import Any, Tuple
from torch import Tensor
import math


# 
BENCHMARK_SPECS = [
    {
        'name': 'adaptec1', 'grid': 128, 'num_macros_to_place': 128, 'rank_mode': 1,
        'wire_mask_scale': 10000.0, 'reward_scale': 1000.0, 'lr_actor': 0.0004, 'lr_critic': 0.001,
        'actor_lr_anneal_rate': 0.9999, 'critic_lr_anneal_rate': 0.9999, 'num_loops': 1000,
        'num_episodes_in_loop': 5, 'num_update_epochs': 10, 'update_batch_size': 128,
        'solution_pool_size': 5, 'update_frontiers_freq': 2, 'update_frontiers_begin': 200,
        'alpha': 0.1, 'gamma': 1.0,
    },
    {
        'name': 'adaptec2', 'grid': 128, 'num_macros_to_place': 128, 'rank_mode': 1,
        'wire_mask_scale': 100000.0, 'reward_scale': 10000.0, 'lr_actor': 0.0003, 'lr_critic': 0.001,
        'actor_lr_anneal_rate': 0.9999, 'critic_lr_anneal_rate': 0.9999, 'num_loops': 1000,
        'num_episodes_in_loop': 5, 'num_update_epochs': 10, 'update_batch_size': 128,
        'solution_pool_size': 5, 'update_frontiers_freq': 2, 'update_frontiers_begin': 200,
        'alpha': 0.3, 'gamma': 1.0,
    },
    {
        'name': 'adaptec3', 'grid': 512, 'num_macros_to_place': 128, 'rank_mode': 2,
        'wire_mask_scale': 10000.0, 'reward_scale': 10000.0, 'lr_actor': 0.0001, 'lr_critic': 0.0001,
        'actor_lr_anneal_rate': 0.9997, 'critic_lr_anneal_rate': 0.9997, 'num_loops': 1000,
        'num_episodes_in_loop': 5, 'num_update_epochs': 10, 'update_batch_size': 128,
        'solution_pool_size': 5, 'update_frontiers_freq': 2, 'update_frontiers_begin': 200,
        'alpha': 0.1, 'gamma': 0.999,
    },
    {
        'name': 'adaptec4', 'grid': 512, 'num_macros_to_place': 128, 'rank_mode': 2,
        'wire_mask_scale': 10000.0, 'reward_scale': 10000.0, 'lr_actor': 0.0001, 'lr_critic': 0.0001,
        'actor_lr_anneal_rate': 0.9997, 'critic_lr_anneal_rate': 0.9997, 'num_loops': 1000,
        'num_episodes_in_loop': 5, 'num_update_epochs': 10, 'update_batch_size': 128,
        'solution_pool_size': 5, 'update_frontiers_freq': 2, 'update_frontiers_begin': 200,
        'alpha': 0.1, 'gamma': 0.999,
    },
    {
        'name': 'bigblue1', 'grid': 512, 'num_macros_to_place': 200, 'rank_mode': 1,
        'wire_mask_scale': 10000.0, 'reward_scale': 1000.0, 'lr_actor': 0.0003, 'lr_critic': 0.0001,
        'actor_lr_anneal_rate': 0.9997, 'critic_lr_anneal_rate': 0.9997, 'num_loops': 1000,
        'num_episodes_in_loop': 5, 'num_update_epochs': 10, 'update_batch_size': 128,
        'solution_pool_size': 5, 'update_frontiers_freq': 2, 'update_frontiers_begin': 200,
        'alpha': 0.1, 'gamma': 0.999,
    },
    {
        'name': 'bigblue2', 'grid': 128, 'num_macros_to_place': 128, 'rank_mode': 2,
        'wire_mask_scale': 10000.0, 'reward_scale': 1000.0, 'lr_actor': 0.0003, 'lr_critic': 0.0001,
        'actor_lr_anneal_rate': 0.9997, 'critic_lr_anneal_rate': 0.9997, 'num_loops': 1000,
        'num_episodes_in_loop': 5, 'num_update_epochs': 10, 'update_batch_size': 128,
        'solution_pool_size': 5, 'update_frontiers_freq': 2, 'update_frontiers_begin': 200,
        'alpha': 0.1, 'gamma': 0.999,
    },
    {
        'name': 'bigblue3', 'grid': 256, 'num_macros_to_place': 128, 'rank_mode': 1,
        'wire_mask_scale': 10000.0, 'reward_scale': 10000.0, 'lr_actor': 0.03, 'lr_critic': 0.001,
        'actor_lr_anneal_rate': 0.9997, 'critic_lr_anneal_rate': 0.9997, 'num_loops': 1000,
        'num_episodes_in_loop': 5, 'num_update_epochs': 10, 'update_batch_size': 128,
        'solution_pool_size': 5, 'update_frontiers_freq': 2, 'update_frontiers_begin': 200,
        'alpha': 0.1, 'gamma': 0.999,
    },
    {
        'name': 'bigblue4', 'grid': 256, 'num_macros_to_place': 128, 'rank_mode': 1,
        'wire_mask_scale': 1000000.0, 'reward_scale': 1000000.0, 'lr_actor': 0.0003, 'lr_critic': 0.001,
        'actor_lr_anneal_rate': 0.9997, 'critic_lr_anneal_rate': 0.9997, 'num_loops': 1000,
        'num_episodes_in_loop': 5, 'num_update_epochs': 10, 'update_batch_size': 128,
        'solution_pool_size': 5, 'update_frontiers_freq': 2, 'update_frontiers_begin': 200,
        'alpha': 0.1, 'gamma': 0.999,
    },
]


# 1. THÊM ĐOẠN NÀY ĐỂ LỌC BENCHMARK (sau này muốn full thì xóa 2 dòng này đi là ok)
TARGET_BENCHMARKS = ['adaptec1', 'adaptec2', 'bigblue1']
BENCHMARK_SPECS   = [spec for spec in BENCHMARK_SPECS if spec['name'] in TARGET_BENCHMARKS]

BENCHMARK_NAMES    = [spec['name'] for spec in BENCHMARK_SPECS]
BENCHMARK_SPEC_MAP = {spec['name']: spec for spec in BENCHMARK_SPECS}
ACTIVE_BENCHMARK   = BENCHMARK_NAMES[0]
DATA_ROOT          = Path('/kaggle/input/datasets/tranvannha/ispd2005/ISPD2005_clean')
OUTPUT_ROOT        = Path('/kaggle/working/efficientplace_runs')
RUN_TAG            = datetime.now().strftime('%Y%m%d-%H%M%S')
RUN_TAG            = RUN_TAG if 'RUN_TAG' in globals() else 'run'


def benchmark_dir_for(name: str) -> Path:
    candidates = [
        DATA_ROOT / 'benchmarks' / 'ispd2005' / name,
        DATA_ROOT / name,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return candidates[0]


BENCHMARK_NAMES = [spec['name'] for spec in BENCHMARK_SPECS]
BENCHMARK_SPEC_MAP = {spec['name']: spec for spec in BENCHMARK_SPECS}
ACTIVE_BENCHMARK = BENCHMARK_NAMES[0]

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('Dataset root exists:', DATA_ROOT.exists())
print('Benchmarks available:', BENCHMARK_NAMES)
print('Example benchmark dir:', benchmark_dir_for(ACTIVE_BENCHMARK))
#Checking Log
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('Run tag:', RUN_TAG)
print('Datasets ready:', all(benchmark_dir_for(name).exists() for name in BENCHMARK_NAMES))

## Bảng thống kê tham số theo benchmark

Bảng dưới đây tóm tắt các tham số đang được dùng trong notebook hiện tại. Nguồn chính của các giá trị này là file cấu hình benchmark trong repo gốc (`conf/benchmark/*.yaml`), sau đó notebook đã được đồng bộ lại theo mô tả trong paper cho các tham số huấn luyện chính như `grid`, `num_loops`, `num_episodes_in_loop`, `num_update_epochs`, `update_frontiers_begin`, `update_frontiers_freq`.

| Benchmark | Grid | Macros | Rank mode | Wire mask scale | Reward scale | LR actor | LR critic | LR anneal | Num loops | Episodes/loop | Update epochs | Frontiers freq | Frontiers begin | Nguồn chính |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---|
| adaptec1 | 128 | 128 | 1 | 10000.0 | 1000.0 | 0.0004 | 0.001 | 0.9999 / 0.9999 | 1000 | 5 | 10 | 2 | 200 | `conf/benchmark/adaptec1.yaml`, đồng bộ theo paper cho grid/training schedule |
| adaptec2 | 128 | 128 | 1 | 100000.0 | 10000.0 | 0.0003 | 0.001 | 0.9999 / 0.9999 | 1000 | 5 | 10 | 2 | 200 | `conf/benchmark/adaptec2.yaml`, đồng bộ theo paper cho episodes/grid |
| adaptec3 | 512 | 128 | 2 | 10000.0 | 10000.0 | 0.0001 | 0.0001 | 0.9997 / 0.9997 | 1000 | 5 | 10 | 2 | 200 | `conf/benchmark/adaptec3.yaml`, đồng bộ theo paper cho training schedule |
| adaptec4 | 512 | 128 | 2 | 10000.0 | 10000.0 | 0.0001 | 0.0001 | 0.9997 / 0.9997 | 1000 | 5 | 10 | 2 | 200 | `conf/benchmark/adaptec4.yaml`, đồng bộ theo paper cho training schedule |
| bigblue1 | 512 | 200 | 1 | 10000.0 | 1000.0 | 0.0003 | 0.0001 | 0.9997 / 0.9997 | 1000 | 5 | 10 | 2 | 200 | `conf/benchmark/bigblue1.yaml`, đồng bộ theo paper cho training schedule |
| bigblue2 | 128 | 128 | 2 | 10000.0 | 1000.0 | 0.0003 | 0.0001 | 0.9997 / 0.9997 | 1000 | 5 | 10 | 2 | 200 | `conf/benchmark/bigblue2.yaml`, notebook giữ giới hạn macro lớn của repo gốc |
| bigblue3 | 256 | 128 | 1 | 10000.0 | 10000.0 | 0.03 | 0.001 | 0.9997 / 0.9997 | 1000 | 5 | 10 | 2 | 200 | `conf/benchmark/bigblue3.yaml`, đồng bộ theo paper cho grid/training schedule |
| bigblue4 | 256 | 128 | 1 | 1000000.0 | 1000000.0 | 0.0003 | 0.001 | 0.9997 / 0.9997 | 1000 | 5 | 10 | 2 | 200 | `conf/benchmark/bigblue4.yaml`, giữ scale lớn như repo gốc |

### Ghi chú nhanh
- Các giá trị `wire_mask_scale`, `reward_scale`, `lr_actor`, `lr_critic`, `actor_lr_anneal_rate`, `critic_lr_anneal_rate`, `rank_mode`, `num_macros_to_place` chủ yếu lấy từ cấu hình benchmark gốc trong repo.
- Các giá trị `grid`, `num_loops`, `num_episodes_in_loop`, `update_frontiers_freq`, `update_frontiers_begin`, `num_update_epochs` đã được đồng bộ lại theo phần tham số huấn luyện của paper trong notebook hiện tại.
- `density weight in phase 2/3` chưa được biểu diễn thành biến riêng trong notebook nên chưa đưa vào bảng này.

# Training

### place_db.py: Read dataset

In [ ]:
def read_node_file(node_file, benchmark):
    fopen = open(node_file, "r")
    node_info = {}
    node_info_raw_id_name ={}
    node_cnt = 0
    for line in fopen.readlines():
        if not line.startswith("\t"):
            continue
        line = line.strip().split()
        if line[-1] != "terminal":
            continue
        node_name = line[0]
        x = int(line[1])
        y = int(line[2])
        node_info[node_name] = {"id": node_cnt, "x": x , "y": y }
        node_info_raw_id_name[node_cnt] = node_name
        node_cnt += 1
        
    assert node_cnt == len(node_info)
    fopen.close()
    
    if benchmark == "bigblue2" or benchmark == "bigblue4":
        node_id_ls = list(node_info.keys()).copy()
        node_area = {}
        for node_id in node_id_ls:
            node_area[node_id] = node_info[node_id]["x"] * node_info[node_id]["y"]
        node_id_ls.sort(key = lambda x: - node_area[x])
        node_id_ls = node_id_ls[:256] if benchmark == "bigblue2" else node_id_ls[:1024]
        node_info = {}
        node_info_tmp = {}
        for node_id in node_id_ls:
            node_info_tmp[node_id] = node_info[node_id]
        node_info = node_info_tmp
    node_cnt = len(node_info)
    
    return node_info, node_info_raw_id_name, node_cnt

def read_net_file(net_file, node_info):
    fopen = open(net_file, "r")
    net_info = {}
    net_name = None
    net_cnt = 0
    for line in fopen.readlines():
        if not line.startswith("\t") and not line.startswith("NetDegree"):
            continue
        line = line.strip().split()
        if line[0] == "NetDegree":
            net_name = line[-1]
        else:
            node_name = line[0]
            if node_name in node_info:
                if not net_name in net_info:
                    net_info[net_name] = {}
                    net_info[net_name]["nodes"] = {}
                    net_info[net_name]["ports"] = {}
                if not node_name in net_info[net_name]["nodes"]:
                    x_offset = float(line[-2])
                    y_offset = float(line[-1])
                    net_info[net_name]["nodes"][node_name] = {}
                    net_info[net_name]["nodes"][node_name] = {"x_offset": x_offset, "y_offset": y_offset}
    for net_name in list(net_info.keys()):
        if len(net_info[net_name]["nodes"]) <= 1:
            net_info.pop(net_name)
    for net_name in net_info:
        net_info[net_name]['id'] = net_cnt
        net_cnt += 1
    
    assert net_cnt == len(net_info)
    fopen.close()
    return net_info, net_cnt

def get_node_to_net_dict(node_info, net_info):
    node_to_net_dict = {}
    for node_name in node_info:
        node_to_net_dict[node_name] = set()
    for net_name in net_info:
        for node_name in net_info[net_name]["nodes"]:
            node_to_net_dict[node_name].add(net_name)
    return node_to_net_dict

def read_pl_file(pl_file, node_info):
    fopen = open(pl_file, "r")
    max_height = 0
    max_width = 0
    for line in fopen.readlines():
        if not line.startswith('o'):
            continue
        line = line.strip().split()
        node_name = line[0]
        if not node_name in node_info:
            continue
        place_x = int(line[1])
        place_y = int(line[2])
        max_height = max(max_height, node_info[node_name]["x"] + place_x)
        max_width = max(max_width, node_info[node_name]["y"] + place_y)
        node_info[node_name]["raw_x"] = place_x
        node_info[node_name]["raw_y"] = place_y
    fopen.close()
    return min(max_height, max_width), min(max_height, max_width)

def rank_macros(placedb, rank_mode: int=1):
    
    node_id_ls = list(placedb.node_info.keys()).copy()
    for node_id in node_id_ls:
        placedb.node_info[node_id]["area"] = placedb.node_info[node_id]["x"] * placedb.node_info[node_id]["y"]
    if placedb.benchmark == "bigblue2" or placedb.benchmark == "bigblue4":
        node_id_ls.sort(key = lambda x: -placedb.node_info[x]["area"])
        return node_id_ls

    net_id_ls = list(placedb.net_info.keys()).copy()
    for net_id in net_id_ls:
        sum = 0
        for node_id in placedb.net_info[net_id]["nodes"].keys():
            sum += placedb.node_info[node_id]["area"]
        placedb.net_info[net_id]["area"] = sum
    for node_id in node_id_ls:
        placedb.node_info[node_id]["area_sum"] = 0
        for net_id in net_id_ls:
            if node_id in placedb.net_info[net_id]["nodes"].keys():
                placedb.node_info[node_id]["area_sum"] += placedb.net_info[net_id]["area"]
    if rank_mode == 1:
        node_id_ls.sort(key = lambda x: (- placedb.node_info[x]["area"], - placedb.node_info[x]["area_sum"]))
    else:
        assert rank_mode == 2
        node_id_ls.sort(key = lambda x: placedb.node_info[x]["area_sum"], reverse = True)
    return node_id_ls

def get_node_id_to_name_topology(node_info, node_to_net_dict, net_info, benchmark):
    node_id_to_name = []
    adjacency = {}
    for net_name in net_info:
        for node_name_1, node_name_2 in list(combinations(net_info[net_name]['nodes'],2)):
            if node_name_1 not in adjacency:
                adjacency[node_name_1] = set()
            if node_name_2 not in adjacency:
                adjacency[node_name_2] = set()
            adjacency[node_name_1].add(node_name_2)
            adjacency[node_name_2].add(node_name_1)

    visited_node = set()

    node_net_num = {}
    for node_name in node_info:
        node_net_num[node_name] = len(node_to_net_dict[node_name])
    
    node_net_num_fea= {}
    node_net_num_max = max(node_net_num.values())
    for node_name in node_info:
        node_net_num_fea[node_name] = node_net_num[node_name]/node_net_num_max
    
    node_area_fea = {}
    node_area_max_node = max(node_info, key = lambda x : node_info[x]['x'] * node_info[x]['y'])
    node_area_max = node_info[node_area_max_node]['x'] * node_info[node_area_max_node]['y']
    for node_name in node_info:
        node_area_fea[node_name] = node_info[node_name]['x'] * node_info[node_name]['y'] / node_area_max
    
    if "V" in node_info:
        add_node = "V"
        visited_node.add(add_node)
        node_id_to_name.append((add_node, node_net_num[add_node]))
        node_net_num.pop(add_node)
    
    add_node = max(node_net_num, key = lambda v: node_net_num[v])
    visited_node.add(add_node)
    node_id_to_name.append((add_node, node_net_num[add_node]))
    node_net_num.pop(add_node)

    while len(node_id_to_name) < len(node_info):
        candidates = {}
        for node_name in visited_node:
            if node_name not in adjacency:
                continue
            for node_name_2 in adjacency[node_name]:
                if node_name_2 in visited_node:
                    continue
                if node_name_2 not in candidates:
                    candidates[node_name_2] = 0
                candidates[node_name_2] += 1
        for node_name in node_info:
            if node_name not in candidates and node_name not in visited_node:
                candidates[node_name] = 0
        if len(candidates) > 0:
            if benchmark != 'ariane':
                if benchmark == "bigblue3":
                    add_node = max(candidates, key = lambda v: candidates[v]*1 + node_net_num[v]*100000 +\
                        node_info[v]['x']*node_info[v]['y'] * 1 +int(hash(v)%10000)*1e-6)
                else:
                    add_node = max(candidates, key = lambda v: candidates[v]*1 + node_net_num[v]*1000 +\
                        node_info[v]['x']*node_info[v]['y'] * 1 +int(hash(v)%10000)*1e-6)
            else:
                add_node = max(candidates, key = lambda v: candidates[v]*30000 + node_net_num[v]*1000 +\
                    node_info[v]['x']*node_info[v]['y']*1 +int(hash(v)%10000)*1e-6)
        else:
            if benchmark != 'ariane':
                if benchmark == "bigblue3":
                    add_node = max(node_net_num, key = lambda v: node_net_num[v]*100000 + node_info[v]['x']*node_info[v]['y']*1)
                else:
                    add_node = max(node_net_num, key = lambda v: node_net_num[v]*1000 + node_info[v]['x']*node_info[v]['y']*1)
            else:
                add_node = max(node_net_num, key = lambda v: node_net_num[v]*1000 + node_info[v]['x']*node_info[v]['y']*1)

        visited_node.add(add_node)
        node_id_to_name.append((add_node, node_net_num[add_node])) 
        node_net_num.pop(add_node)
    for i, (node_name, _) in enumerate(node_id_to_name):
        node_info[node_name]["id"] = i
    node_id_to_name_res = [x for x, _ in node_id_to_name]
    return node_id_to_name_res

class PlaceDB():
    def __init__(self, benchmark, benchmark_dir, rank_mode: int=1):
        self.benchmark = benchmark
        self.benchmark_dir = benchmark_dir
        assert os.path.exists(benchmark_dir)

        # Read node file
        node_file = os.path.join(benchmark_dir, benchmark + ".nodes")
        self.node_info, self.node_info_raw_id_name, self.node_cnt = read_node_file(node_file, benchmark)

        # Read net file
        net_file = os.path.join(benchmark_dir, benchmark+".nets")
        self.net_info, self.net_cnt = read_net_file(net_file, self.node_info)
        self.node_to_net_dict = get_node_to_net_dict(self.node_info, self.net_info)
        
        # Read pl file
        pl_file = os.path.join(benchmark_dir, benchmark + ".pl")
        if benchmark == "bigblue1":
            self.max_height, self.max_width = 12800, 12800
        if benchmark == "bigblue3":
            self.max_height, self.max_width = 30464, 30464
        else:
            self.max_height, self.max_width = read_pl_file(pl_file, self.node_info)

        # Rank the macros according to the rank_mode
        if rank_mode == 3:
            self.node_id_to_name = get_node_id_to_name_topology(self.node_info, self.node_to_net_dict, self.net_info, self.benchmark)
        else:
            self.node_id_to_name = rank_macros(self, rank_mode)


### agent.py: CORE - nhận trạng thái placement hiện tại và sinh ra P(x) trên toàn bộ grid để quyết định đặt macro ở đâu tiếp theo.

In [ ]:
class Actor(nn.Module):
    def __init__(self, num_macros_to_place: int, grid: int):
        super().__init__()
        self.grid = grid

        self.conv_1 = nn.Sequential(
            nn.AvgPool2d(kernel_size=int(grid / 128), stride=int(grid / 128)),
            nn.Conv2d(3, 8, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
        )
        self.pool_1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv_2 = nn.Sequential(
            nn.Conv2d(8, 16, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
        )
        self.pool_2 = nn.MaxPool2d(kernel_size=4, stride=4)

        self.conv_3 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
        )
        self.pool_3 = nn.MaxPool2d(kernel_size=4, stride=4)

        self.conv_4 = nn.Sequential(
            nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
        )

        self.fc = nn.Sequential(
            nn.Linear(4 * 4 * 32 + 32, 512),
            nn.ReLU(),
            nn.Linear(512, 4 * 4 * 32),
            nn.ReLU(),
        )

        self.up_5 = nn.Upsample(scale_factor=4, mode='bilinear', align_corners=True)
        self.conv_5 = nn.Sequential(
            nn.Conv2d(64, 16, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
        )

        self.up_6 = nn.Upsample(scale_factor=4, mode='bilinear', align_corners=True)
        self.conv_6 = nn.Sequential(
            nn.Conv2d(32, 8, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
        )

        self.up_7 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv_7 = nn.Sequential(
            nn.Conv2d(16, 8, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
        )

        self.conv_8 = nn.Sequential(
            nn.Upsample(scale_factor=int(grid / 128), mode='bilinear', align_corners=True),
            nn.Conv2d(8, 1, kernel_size=1, stride=1),
        )

        self.time_embedding = nn.Embedding(num_macros_to_place, 32)
    
    def __call__(self, s, t, *args: Any, **kwds: Any) -> Tuple[Distribution, Tensor, Tensor]:
        logits = self.forward(s, t)
        distr, pi_distr, action_distr, pi_distr_nomask = self.get_distr(logits, position_mask=kwds['position_mask'], wire_mask=kwds['wire_mask'])
        return distr, pi_distr, action_distr, pi_distr_nomask

    def forward(self, s, t):   # [B, 3, 256, 256]
        
        feature_1 = self.conv_1(s)  # [B, 8, 256, 256]
        feature_1_pool = self.pool_1(feature_1) # [B, 8, 64, 64]

        feature_2 = self.conv_2(feature_1_pool) # [B, 16, 128, 128]
        feature_2_pool = self.pool_2(feature_2) # [B, 16, 16, 16]

        feature_3 = self.conv_3(feature_2_pool) # [B, 32, 16, 16]
        feature_3_pool = self.pool_3(feature_3) # [B, 32, 4, 4]

        feature_4 = self.conv_4(feature_3_pool) # [B, 32, 4, 4]

        time_embedding = self.time_embedding(t) # [B, 32]
        feature = torch.cat([torch.reshape(feature_4, [-1, 4 * 4 * 32]), time_embedding], dim=-1) # [B, 544]
        feature = torch.reshape(self.fc(feature), [-1, 32, 4, 4])

        feature_5 = self.up_5(feature)    # [B, 32, 16, 16]
        feature_5 = self.conv_5(torch.cat((feature_3, feature_5), dim=1))   # [B, 16, 16, 16]

        feature_6 = self.up_6(feature_5)    # [B, 16, 64, 64]
        feature_6 = self.conv_6(torch.cat((feature_2, feature_6), dim=1))   # [B, 16, 64, 64]

        feature_7 = self.up_7(feature_6)   # [B, 8, 128, 128]
        feature_7 = self.conv_7(torch.cat((feature_1, feature_7), dim=1))   # [B, 8, 128, 128]

        feature_8 = self.conv_8(feature_7)   # [B, 1, 128, 128]
        
        return torch.reshape(feature_8, [-1, self.grid * self.grid])

    def get_distr(self, logits: Tensor, position_mask: Tensor, wire_mask: Tensor):
        mask = torch.where(position_mask > 0.5, torch.inf, wire_mask).reshape([-1, self.grid * self.grid])
        wire_min = mask.min(dim=-1, keepdim=True)[0]
        greedy_mask = torch.where(mask == wire_min, 0.0, - 1e10).view([-1, self.grid * self.grid])

        action_mask = torch.reshape(position_mask >= 0.5, [-1, self.grid * self.grid])
        pi_distr = torch.where(action_mask, - 1e10, logits)
        greedy_distr = pi_distr + greedy_mask
        pi_distr = torch.softmax(pi_distr, dim=-1)
        
        pi_distr_no_mask = torch.softmax(logits, dim=-1)
        
        greedy_distr = torch.softmax(greedy_distr, dim=-1)

        action_distr = greedy_distr
        distr = Categorical(action_distr)

        return distr, pi_distr, action_distr, pi_distr_no_mask


### replay_buffer.py: Memory để lưu dữ liệu kinh nghiệm khi train RL



In [ ]:
class ReplayBuffer:
    def __init__(self, capacity, grid):
        self.s = torch.zeros([capacity, 3, grid, grid])
        self.t = torch.zeros([capacity], dtype=torch.long)
        self.a = torch.zeros([capacity, 1], dtype=torch.long)
        self.a_logp = torch.zeros([capacity, 1])
        self.r = torch.zeros([capacity, 1])
        self.s_ = torch.zeros([capacity, 3, grid, grid])
        self.t_ = torch.zeros([capacity], dtype=torch.long)
        self.dw = torch.zeros([capacity, 1])
        self.done = torch.zeros([capacity, 1])
        self.capacity = capacity
        self.count = 0

    def store(self, s, t, a, a_logp, r, s_, t_, done):
        assert self.count <= self.capacity
        self.s[self.count] = s
        self.t[self.count] = t
        self.a[self.count] = a
        self.a_logp[self.count] = a_logp
        self.r[self.count] = r
        self.s_[self.count] = s_
        self.t_[self.count] = t_
        self.done[self.count] = done
        self.count += 1

    def get_dataset(self, device):
        s = self.s[:self.count].to(device)
        t = self.t[:self.count].to(device)
        a = self.a[:self.count].to(device)
        a_logp = self.a_logp[:self.count].to(device)
        r = self.r[:self.count].to(device)
        s_ = self.s_[:self.count].to(device)
        t_ = self.t_[:self.count].to(device)
        done = self.done[:self.count].to(device)

        self.count = 0
        return s, t, a, a_logp, r, s_, t_, done

### Environment.py: Biến bài toán macro placement thành môi trường RL, nơi agent đặt từng macro và nhận reward theo chất lượng placement. 
Hay tức là nơi agent tương tác để đặt macro từng bước.

In [ ]:
class Environment:
    def __init__(self, benchmark, benchmark_dir, rank_mode, wire_mask_scale, reward_scale, num_macros_to_place, grid,):

        placedb = PlaceDB(benchmark, benchmark_dir, rank_mode)
        self.placedb = placedb

        self.max_height = placedb.max_height
        self.max_width = placedb.max_width
        
        self.num_macros = placedb.node_cnt
        self.num_nets = placedb.net_cnt
        self.macro_name_list = placedb.node_id_to_name
        self.macros = [self.placedb.node_info[self.macro_name_list[i]]
                       for i in range(self.num_macros)]
        
        self.num_macros_to_place = num_macros_to_place
        self.grid = grid
        self.ratio = self.max_height / self.grid
        
        self.size_x = [max(1, math.ceil(self.placedb.node_info[node_name]
                           ['x'] / self.ratio)) for node_name in self.macro_name_list]
        self.size_y = [max(1, math.ceil(self.placedb.node_info[node_name]
                           ['y'] / self.ratio)) for node_name in self.macro_name_list]

        self.wire_mask_scale = wire_mask_scale
        self.reward_scale = reward_scale

    def reset(self):
        self.t = 0
        self.macro_pos = {}
        self.net_bound_info = {}

        canvas = torch.zeros((self.grid, self.grid))
        wire_mask = torch.zeros((self.grid, self.grid))
        next_x = max(1, math.ceil(self.macros[0]['x'] / self.ratio))
        next_y = max(1, math.ceil(self.macros[0]['y'] / self.ratio))
        position_mask = self.calc_position_mask(canvas, next_x, next_y)

        self.state = torch.cat([
            canvas.unsqueeze(0),
            wire_mask.unsqueeze(0),
            position_mask.unsqueeze(0),
        ], dim=0)

        return self.state

    def step(self, action):
        canvas, wire_mask, position_mask = self.state[0], self.state[1], self.state[2]

        pos_x = round(action // self.grid)
        pos_y = round(action % self.grid)
        size_x = self.size_x[self.t]
        size_y = self.size_y[self.t]

        # place the macro
        canvas[pos_x: pos_x + size_x, pos_y: pos_y + size_y] = 1.0
        canvas[pos_x: pos_x + size_x, pos_y] = 0.5
        if pos_y + size_y - 1 < self.grid:
            canvas[pos_x: pos_x + size_x, max(0, pos_y + size_y - 1)] = 0.5
        canvas[pos_x, pos_y: pos_y + size_y] = 0.5
        if pos_x + size_x - 1 < self.grid:
            canvas[max(0, pos_x + size_x - 1), pos_y: pos_y + size_y] = 0.5
        self.macro_pos[self.macro_name_list[self.t]] = (pos_x, pos_y, size_x, size_y)

        # calculate the reward
        hpwl_increment = wire_mask[pos_x, pos_y] * self.wire_mask_scale
        reward = - hpwl_increment / self.reward_scale

        if position_mask[pos_x][pos_y] == 1:
            reward = -1
            logging.info("INVALID ACTION.")

        node_name = self.macro_name_list[self.t]
        for net_name in self.placedb.node_to_net_dict[node_name]:
            pin_x = round((pos_x * self.ratio + self.placedb.node_info[node_name]['x'] / 2 +
                           self.placedb.net_info[net_name]["nodes"][node_name]["x_offset"]) / self.ratio)
            pin_y = round((pos_y * self.ratio + self.placedb.node_info[node_name]['y'] / 2 +
                           self.placedb.net_info[net_name]["nodes"][node_name]["y_offset"]) / self.ratio)
            if net_name in self.net_bound_info:
                self.net_bound_info[net_name]['max_x'] = max(
                    pin_x, self.net_bound_info[net_name]['max_x'])
                self.net_bound_info[net_name]['min_x'] = min(
                    pin_x, self.net_bound_info[net_name]['min_x'])
                self.net_bound_info[net_name]['max_y'] = max(
                    pin_y, self.net_bound_info[net_name]['max_y'])
                self.net_bound_info[net_name]['min_y'] = min(
                    pin_y, self.net_bound_info[net_name]['min_y'])
            else:
                self.net_bound_info[net_name] = {}
                self.net_bound_info[net_name]['max_x'] = pin_x
                self.net_bound_info[net_name]['min_x'] = pin_x
                self.net_bound_info[net_name]['max_y'] = pin_y
                self.net_bound_info[net_name]['min_y'] = pin_y

        self.t += 1
        done = self.is_done()

        if self.t < self.num_macros:
            next_x = math.ceil(max(
                1, self.placedb.node_info[self.placedb.node_id_to_name[self.t]]['x']/self.ratio))
            next_y = math.ceil(max(
                1, self.placedb.node_info[self.placedb.node_id_to_name[self.t]]['y']/self.ratio))
            position_mask = self.calc_position_mask(canvas, next_x, next_y)
            wire_mask = self.calc_wiremask()
        else:
            next_x = 0
            next_y = 0
            position_mask = self.calc_position_mask(canvas, next_x, next_y)
            wire_mask = torch.zeros((self.grid, self.grid))

        next_state = torch.cat([
            canvas.unsqueeze(0),
            wire_mask.unsqueeze(0),
            position_mask.unsqueeze(0),
        ], dim=0).float()

        self.state = next_state
        info = {
            "delta_hpwl": hpwl_increment,
            "wire_mask": wire_mask,
            "position_mask": position_mask
        }

        return next_state, reward, done, info

    def calc_position_mask(self, canvas, next_x, next_y):
        mask = torch.zeros((self.grid, self.grid))
        for node_name in self.macro_pos:
            startx = max(0, self.macro_pos[node_name][0] - next_x + 1)
            starty = max(0, self.macro_pos[node_name][1] - next_y + 1)
            endx = min(self.macro_pos[node_name][0] +
                       self.macro_pos[node_name][2] - 1, self.grid - 1)
            endy = min(self.macro_pos[node_name][1] +
                       self.macro_pos[node_name][3] - 1, self.grid - 1)
            mask[startx: endx + 1, starty: endy + 1] = 1
        mask[self.grid - next_x + 1:, :] = 1
        mask[:, self.grid - next_y + 1:] = 1
        return mask

    def calc_wiremask(self):
        wire_mask = torch.zeros((self.grid, self.grid))
        node_name = self.placedb.node_id_to_name[self.t]

        for net_name in self.placedb.node_to_net_dict[node_name]:
            if net_name in self.net_bound_info:
                delta_pin_x = round((self.placedb.node_info[node_name]['x']/2 +
                                     self.placedb.net_info[net_name]["nodes"][node_name]["x_offset"])/self.ratio)
                delta_pin_y = round((self.placedb.node_info[node_name]['y']/2 +
                                     self.placedb.net_info[net_name]["nodes"][node_name]["y_offset"])/self.ratio)
                start_x = self.net_bound_info[net_name]['min_x'] - delta_pin_x
                end_x = self.net_bound_info[net_name]['max_x'] - delta_pin_x
                start_y = self.net_bound_info[net_name]['min_y'] - delta_pin_y
                end_y = self.net_bound_info[net_name]['max_y'] - delta_pin_y

                wire_mask_x = torch.arange(self.grid, dtype=torch.float).unsqueeze(1).repeat(1, self.grid)
                wire_mask_x[:max(start_x, 0)] = start_x - wire_mask_x[:max(start_x, 0)]
                wire_mask_x[max(start_x, 0): max(end_x + 1, 0)] = 0.0
                wire_mask_x[max(end_x + 1, 0):] = wire_mask_x[max(end_x + 1, 0):] - end_x

                wire_mask_y = torch.arange(self.grid, dtype=torch.float).unsqueeze(0).repeat(self.grid, 1)
                wire_mask_y[:, :max(start_y, 0)] = start_y - wire_mask_y[:, :max(start_y, 0)]
                wire_mask_y[:, max(start_y, 0): max(end_y + 1, 0)] = 0.0
                wire_mask_y[:, max(end_y + 1, 0):] = wire_mask_y[:, max(end_y + 1, 0):] - end_y

                wire_mask += wire_mask_x + wire_mask_y

        return wire_mask / self.wire_mask_scale

    def is_done(self):
        return self.t >= self.num_macros or self.t >= self.num_macros_to_place


### main.py

In [ ]:
class SolutionPool:
    def __init__(self, pool_size: int, alpha: float) -> None:
        self.pool_size = pool_size
        self.alpha = alpha

        self.best_hpwl = 1e9
        self.best_hpwl_max = 1e9

        self.sols = []
        self.tree = treelib.Tree()
        self.tree.create_node(
            tag="root",
            identifier=str([]),
            data={
                "visit_time": 0,
                "hpwl": 1e9,
                "hist_hpwl": {},
                "score": 0.0
            }
        )
        self.frontiers = [[]]
        self.frontiers_hist = [[]]

    @property
    def best_solution(self):
        return self.sols[0]["placement"], self.sols[0]["solution"], self.sols[0]["hpwl"]
    
    @property
    def depth_min_max(self):
        depth = [len(x) for x in self.frontiers]
        return min(depth), max(depth)
    
    def update_sol(self, frontier_id, solution, hpwl, placement):
        if hpwl < self.best_hpwl_max:
            sol_record = {
                "solution": solution,
                "placement": placement,
                "hpwl": hpwl,
            }
            self.sols.append(sol_record)
            self.sols = sorted(self.sols, key=lambda x: x["hpwl"])
            self.sols = self.sols[:self.pool_size]

            self.best_hpwl_max = self.sols[-1]["hpwl"]
            self.best_hpwl = self.sols[0]["hpwl"]

        for i in range(len(solution)):
            node = self.tree.get_node(str(solution[:i]))
            if node:
                node.data["visit_time"] += 1
                node.data["score"] = - node.data["hpwl"] + self.alpha * 1 / np.sqrt(node.data["visit_time"])
                
                if i < len(solution) - 1 and solution[:i+1] not in self.frontiers_hist:
                    node.data["hist_hpwl"][str(solution[i])] = hpwl.item() if str(
                        solution[i]) not in node.data["hist_hpwl"] else min(node.data["hist_hpwl"][str(solution[i])], hpwl.item())
                    node.data["hpwl"] = min(node.data["hpwl"], hpwl.item())
                    
            if node is None:
                self.tree.create_node(
                    tag=str(solution[i-1]),
                    identifier=str(solution[:i]),
                    parent=str(solution[:i-1]),
                    data={
                        "visit_time": 1,
                        "hpwl": hpwl.item(),
                        "score": - hpwl.item() + self.alpha * 1 / np.sqrt(1),
                        "hist_hpwl": {str(solution[i]): hpwl.item()},
                    }
                )

    def update_frontiers(self):
        front_scores = [(self.tree.get_node(str(x)).data["score"], x)
                        for x in self.frontiers]
        front_scores = sorted(
            front_scores, key=lambda x: self.tree.get_node(str(x[1])).data["hpwl"])
        assert len(front_scores) <= self.pool_size

        self.frontiers_cand = []
        for front in self.frontiers:
            for chid in self.tree.children(str(front)):
                if eval(chid.identifier) not in self.frontiers and eval(chid.identifier) not in self.frontiers_cand:
                    self.frontiers_cand.append(eval(chid.identifier))
        cand_scores = [(self.tree.get_node(str(x)).data["score"], x)
                       for x in self.frontiers_cand]
        cand_scores = sorted(
            cand_scores, key=lambda x: self.tree.get_node(str(x[1])).data["hpwl"])
        while len(cand_scores) > 0 and (cand_scores[0][0] >= front_scores[-1][0] or len(front_scores) < self.pool_size):
            node = self.tree.get_node(str(cand_scores[0][1]))
            parent = self.tree.parent(node.identifier)
            if node.tag in parent.data["hist_hpwl"]:
                parent.data["hist_hpwl"].pop(node.tag)
            parent.data["hpwl"] = 1e9 if len(parent.data["hist_hpwl"].values(
            )) == 0 else min(parent.data["hist_hpwl"].values())
            parent.data["score"] = - parent.data["hpwl"] + \
                self.alpha * 1 / np.sqrt(parent.data["visit_time"])

            front_scores.append(cand_scores[0])
            front_scores = sorted(
                front_scores, key=lambda x: x[0], reverse=True)
            self.frontiers.append(cand_scores[0][1])
            self.frontiers_hist.append(cand_scores[0][1])
            if len(self.frontiers) > self.pool_size:
                self.frontiers.remove(front_scores[-1][1])
                front_scores = front_scores[:-1]
            self.frontiers_cand.remove(cand_scores[0][1])
            cand_scores = cand_scores[1:]

    def sample(self):
        frontier_id = np.random.choice(len(self.frontiers))
        return frontier_id, self.frontiers[frontier_id]


In [ ]:
class Critic(nn.Module):
    def __init__(self, num_macros_to_place, grid):
        super().__init__()

        self.conv_1 = nn.Sequential(
            nn.AvgPool2d(kernel_size=int(grid / 128), stride=int(grid / 128)),
            nn.Conv2d(3, 8, kernel_size=3, stride=1, padding=1),
            nn.GELU(),
        )
        self.pool_1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv_2 = nn.Sequential(
            nn.Conv2d(8, 16, kernel_size=3, stride=1, padding=1),
            nn.GELU(),
        )
        self.pool_2 = nn.MaxPool2d(kernel_size=4, stride=4)

        self.conv_3 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.GELU(),
        )
        self.pool_3 = nn.MaxPool2d(kernel_size=4, stride=4)

        self.conv_4 = nn.Sequential(
            nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=1),
            nn.GELU(),
        )

        self.fc1 = nn.Sequential(
            nn.Linear(4 * 4 * 32 + 32, 512),
            nn.Tanh(),
            nn.Linear(512, 512),
        )

        self.time_embedding = nn.Embedding(num_macros_to_place + 1, 32)

        self.fc2 = nn.Sequential(
            nn.Linear(512, 512),
            nn.Tanh(),
            nn.Linear(512, 1),
        )

        self.time_value = nn.Embedding(num_macros_to_place + 1, 1)

    def __call__(self, *args: Any, **kwds: Any) -> Tensor:
        return self.forward(*args, **kwds)
    
    def forward(self, s: Tensor, t: Tensor):   # [B, 3, 128, 128]

        feature_1 = self.conv_1(s)  # [B, 8, 256, 256]
        feature_1_pool = self.pool_1(feature_1) # [B, 8, 64, 64]

        feature_2 = self.conv_2(feature_1_pool) # [B, 16, 128, 128]
        feature_2_pool = self.pool_2(feature_2) # [B, 16, 16, 16]

        feature_3 = self.conv_3(feature_2_pool) # [B, 32, 16, 16]
        feature_3_pool = self.pool_3(feature_3) # [B, 32, 4, 4]

        feature_4 = self.conv_4(feature_3_pool) # [B, 32, 4, 4]

        time_embedding = self.time_embedding(t) # [B, 32]
        feature = torch.cat([torch.reshape(feature_4, [-1, 4 * 4 * 32]), time_embedding], dim=-1) # [B, 544]

        feature = self.fc1(feature)

        value = self.fc2(feature) + self.time_value(t) # [B, 1]

        return value

In [ ]:
class Actor(nn.Module):
    def __init__(self, num_macros_to_place: int, grid: int):
        super().__init__()

        self.grid = grid

        self.conv_1 = nn.Sequential(
            nn.AvgPool2d(kernel_size=int(grid / 128), stride=int(grid / 128)),
            nn.Conv2d(3, 8, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
        )
        self.pool_1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv_2 = nn.Sequential(
            nn.Conv2d(8, 16, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
        )
        self.pool_2 = nn.MaxPool2d(kernel_size=4, stride=4)

        self.conv_3 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
        )
        self.pool_3 = nn.MaxPool2d(kernel_size=4, stride=4)

        self.conv_4 = nn.Sequential(
            nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
        )

        self.fc = nn.Sequential(
            nn.Linear(4 * 4 * 32 + 32, 512),
            nn.ReLU(),
            nn.Linear(512, 4 * 4 * 32),
            nn.ReLU(),
        )

        self.up_5 = nn.Upsample(scale_factor=4, mode='bilinear', align_corners=True)
        self.conv_5 = nn.Sequential(
            nn.Conv2d(64, 16, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
        )

        self.up_6 = nn.Upsample(scale_factor=4, mode='bilinear', align_corners=True)
        self.conv_6 = nn.Sequential(
            nn.Conv2d(32, 8, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
        )

        self.up_7 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv_7 = nn.Sequential(
            nn.Conv2d(16, 8, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
        )

        self.conv_8 = nn.Sequential(
            nn.Upsample(scale_factor=int(grid / 128), mode='bilinear', align_corners=True),
            nn.Conv2d(8, 1, kernel_size=1, stride=1),
        )

        self.time_embedding = nn.Embedding(num_macros_to_place, 32)
    
    def __call__(self, s, t, *args: Any, **kwds: Any) -> Tuple[Distribution, Tensor, Tensor]:
        logits = self.forward(s, t)
        distr, pi_distr, action_distr, pi_distr_nomask = self.get_distr(logits, position_mask=kwds['position_mask'], wire_mask=kwds['wire_mask'])
        return distr, pi_distr, action_distr, pi_distr_nomask

    def forward(self, s, t):   # [B, 3, 256, 256]
        
        feature_1 = self.conv_1(s)  # [B, 8, 256, 256]
        feature_1_pool = self.pool_1(feature_1) # [B, 8, 64, 64]

        feature_2 = self.conv_2(feature_1_pool) # [B, 16, 128, 128]
        feature_2_pool = self.pool_2(feature_2) # [B, 16, 16, 16]

        feature_3 = self.conv_3(feature_2_pool) # [B, 32, 16, 16]
        feature_3_pool = self.pool_3(feature_3) # [B, 32, 4, 4]

        feature_4 = self.conv_4(feature_3_pool) # [B, 32, 4, 4]

        time_embedding = self.time_embedding(t) # [B, 32]
        feature = torch.cat([torch.reshape(feature_4, [-1, 4 * 4 * 32]), time_embedding], dim=-1) # [B, 544]
        feature = torch.reshape(self.fc(feature), [-1, 32, 4, 4])

        feature_5 = self.up_5(feature)    # [B, 32, 16, 16]
        feature_5 = self.conv_5(torch.cat((feature_3, feature_5), dim=1))   # [B, 16, 16, 16]

        feature_6 = self.up_6(feature_5)    # [B, 16, 64, 64]
        feature_6 = self.conv_6(torch.cat((feature_2, feature_6), dim=1))   # [B, 16, 64, 64]

        feature_7 = self.up_7(feature_6)   # [B, 8, 128, 128]
        feature_7 = self.conv_7(torch.cat((feature_1, feature_7), dim=1))   # [B, 8, 128, 128]

        feature_8 = self.conv_8(feature_7)   # [B, 1, 128, 128]
        
        return torch.reshape(feature_8, [-1, self.grid * self.grid])

    def get_distr(self,
        logits: Tensor,
        position_mask: Tensor, wire_mask: Tensor
    ):

        mask = torch.where(position_mask > 0.5, torch.inf, wire_mask).reshape([-1, self.grid * self.grid])
        wire_min = mask.min(dim=-1, keepdim=True)[0]
        greedy_mask = torch.where(mask == wire_min, 0.0, - 1e10).view([-1, self.grid * self.grid])

        action_mask = torch.reshape(position_mask >= 0.5, [-1, self.grid * self.grid])
        pi_distr = torch.where(action_mask, - 1e10, logits)
        greedy_distr = pi_distr + greedy_mask
        pi_distr = torch.softmax(pi_distr, dim=-1)
        
        pi_distr_no_mask = torch.softmax(logits, dim=-1)
        
        greedy_distr = torch.softmax(greedy_distr, dim=-1)

        action_distr = greedy_distr
        distr = Categorical(action_distr)

        return distr, pi_distr, action_distr, pi_distr_no_mask


In [ ]:
@hydra.main(version_base=None, config_path="conf", config_name="main")
def main(config: DictConfig):
    # set random seed
    random.seed(config.seed)
    np.random.seed(config.seed)
    torch.manual_seed(config.seed)
    torch.cuda.manual_seed(config.seed)
    
    # set cpu num
    cpu_num = 10
    os.environ['OMP_NUM_THREADS'] = str(cpu_num)
    os.environ['OPENBLAS_NUM_THREADS'] = str(cpu_num)
    os.environ['MKL_NUM_THREADS'] = str(cpu_num)
    os.environ['VECLIB_MAXIMUM_THREADS'] = str(cpu_num)
    os.environ['NUMEXPR_NUM_THREADS'] = str(cpu_num)
    torch.set_num_threads(cpu_num)
    
    # set device
    if(torch.cuda.is_available()): 
        device = torch.device(f'cuda:{config.cuda}')
        torch.cuda.empty_cache()
        logging.info(f'Using GPU: {torch.cuda.get_device_name()}, cuda: {config.cuda}.')
    else:
        device = torch.device('cpu')
        logging.info('Using CPU.')
    
    # set up tensorboard writer
    tb_writer = SummaryWriter(config.tb_dir)

    # instantiate agent, environment and trainer
    agent: Agent = hydra.utils.instantiate(config.agent)
    agent.set_up(device, tb_writer)
    env: Environment = hydra.utils.instantiate(config.env)
    trainer: Trainer = hydra.utils.instantiate(config.trainer)
    trainer.setup_tb_writer(tb_writer)

    # train the model
    trainer.train(agent, env)

In [ ]:
class Agent:
    def __init__(self, max_grad_norm, clip_epsilon, entropy_coef, lr_actor, lr_critic,
        actor_lr_anneal_rate, critic_lr_anneal_rate, num_macros_to_place, grid, lamda, gamma=0.99,):
        super().__init__()

        self.actor_net = Actor(num_macros_to_place, grid)
        self.critic_net = Critic(num_macros_to_place, grid)

        self.actor_optimizer = torch.optim.Adam(self.actor_net.parameters(), lr_actor)
        
        self.critic_optimizer = torch.optim.Adam(self.critic_net.parameters(), lr_critic)
        
        self.actor_lr_scheduler = optim.lr_scheduler.ExponentialLR(self.actor_optimizer, actor_lr_anneal_rate)
        self.critic_lr_scheduler = optim.lr_scheduler.ExponentialLR(self.critic_optimizer, critic_lr_anneal_rate)
        
        self.training_step = 0
        self.lamda = lamda

        self.max_grad_norm = max_grad_norm
        self.clip_epsilon = clip_epsilon
        self.entropy_coef = entropy_coef
        self.gamma = gamma
        self.grid = grid

    
    def set_up(self, device, tb_writer: SummaryWriter):
        self.device = device
        self.actor_net.to(device)
        self.critic_net.to(device)
        gird_X, grid_Y = torch.meshgrid(torch.linspace(0, 1, self.grid+2)[1:-1], torch.linspace(0, 1, self.grid+2)[1:-1], indexing='xy')
        self.grid_points = torch.stack([gird_X, grid_Y], dim=-1).reshape(-1, 2).to(device)
        self.tb_writer = tb_writer
        self.actor_net.grid_points = self.grid_points
    
    @torch.no_grad()
    def select_action(self, state: Tensor, time_step:int):
        
        state = state.unsqueeze(0).to(self.device)
        canvas = state[:,0]
        wire_mask = state[:,1]
        position_mask = state[:,2]
        time_step = torch.LongTensor([time_step]).to(self.device)

        self.actor_net.eval()
        distr, pi_prob, a_prob, pi_prob_nomask = self.actor_net(state, time_step, position_mask=position_mask, wire_mask=wire_mask)
        action = distr.sample()
        action_log_prob = distr.log_prob(action.squeeze()) #a_prob.log().flatten()[action]

        return action.item(), action_log_prob.item()

    @torch.no_grad()
    def act_greedy(self, state: Tensor):
        wire_mask = state[1].float().to(self.device).unsqueeze(0)
        position_mask = state[2].int().to(self.device).unsqueeze(0)
        mask = torch.where(position_mask > 0.5, torch.inf, wire_mask).reshape(self.grid * self.grid)
        wire_min = mask.min()
        action_cand = torch.where(mask == wire_min)[0]
        action_idx = torch.randint(len(action_cand), (1,)).item()
        return action_cand[action_idx].item()

    def get_value(self, state):
        state = torch.from_numpy(state)
        with torch.no_grad():
            value = self.critic_net(state)
        return value.item()

    def update(self, replay_buffer: ReplayBuffer, num_update_epochs: int, batch_size: int):
        """Update the agent's actor and critic networks by sampling from the replay buffer.

        Args:
            replay_buffer (ReplayBuffer): The replay buffer to sample experiences from.
            num_update_epochs (int): The number of epochs to use for updating the networks.
            batch_size (int): The batch size to use for updating the networks.
        """
        
        s, t, a, a_logp, r, s_, t_, done = replay_buffer.get_dataset(self.device)

        adv = []
        self.critic_net.eval()
        with torch.no_grad():
            v_target_list = []
            target = 0
            for re, d in zip(reversed(r.flatten().cpu().numpy()), reversed(done.flatten().cpu().numpy())):
                if d: target = 0
                target = re + self.gamma * target
                v_target_list.append(target)
            v_target_list.reverse()
            v_target = torch.tensor(v_target_list, dtype=torch.float).view(-1, 1).to(self.device)
            adv = v_target - self.critic_net(s, t)
            adv = ((adv - adv.mean()) / (adv.std() + 1e-5))

            distr, pi_prob, a_prob, pi_prob_nomask = self.actor_net(s, t, position_mask=s[:,2], wire_mask=s[:,1])
            a_logp = distr.log_prob(a.squeeze()).unsqueeze(1)


        self.actor_net.train()
        self.critic_net.train()
        for _ in range(num_update_epochs):

            for idx in BatchSampler(SubsetRandomSampler(range(len(s))), batch_size, False):

                distr, pi_prob, a_prob,pi_prob_nomask = self.actor_net(s[idx], t[idx], position_mask=s[idx][:,2], wire_mask=s[idx][:,1])
                distr_entropy = torch.mean(distr.entropy())
                a_logp_new = distr.log_prob(a[idx].squeeze()).unsqueeze(1)

                ratios = torch.exp(torch.clamp(a_logp_new - a_logp[idx], max=7))
                surr1 = ratios * adv[idx]
                surr2 = torch.clamp(ratios, 1 - self.clip_epsilon, 1 + self.clip_epsilon) * adv[idx]
                actor_loss = - torch.min(surr1, surr2).mean()
                self.actor_optimizer.zero_grad()
                actor_loss.backward()
                torch.nn.utils.clip_grad_norm_(self.actor_net.parameters(), self.max_grad_norm)
                self.actor_optimizer.step()

                v_s = self.critic_net(s[idx], t[idx])
                critic_loss = F.smooth_l1_loss(v_s, v_target[idx])
                self.critic_optimizer.zero_grad()
                critic_loss.backward()
                nn.utils.clip_grad_norm_(self.critic_net.parameters(), self.max_grad_norm)
                self.critic_optimizer.step()

                self.training_step += 1

                self.actor_lr_scheduler.step()
                self.critic_lr_scheduler.step()

                self.tb_writer.add_scalar('Loss/actor_loss', actor_loss, self.training_step)
                self.tb_writer.add_scalar('Loss/critic_loss', critic_loss.mean(), self.training_step)
                self.tb_writer.add_scalar('Loss/entropy', distr_entropy, self.training_step)
                self.tb_writer.add_scalar('Params/lr_actor', self.actor_lr_scheduler.get_last_lr()[0], self.training_step)
                self.tb_writer.add_scalar('Params/lr_critic', self.critic_lr_scheduler.get_last_lr()[0], self.training_step)

In [ ]:
class Trainer:
    def __init__(self, num_loops, num_episodes_in_loop, num_update_epochs, update_batch_size, num_macros_to_place, 
                 solution_pool_size, alpha, update_frontiers_begin, update_frontiers_freq, model_dir, solution_dir):

        self.num_loops = num_loops
        self.num_episodes_in_loop = num_episodes_in_loop

        self.num_update_epochs = num_update_epochs
        self.update_batch_size = update_batch_size
        self.num_macros_to_place = num_macros_to_place

        self.buffer_capacity = num_episodes_in_loop * num_macros_to_place
        self.solution_pool_size = solution_pool_size
        self.alpha = alpha

        self.update_frontiers_begin = update_frontiers_begin
        self.update_frontiers_freq = update_frontiers_freq
        self.solution_pool = SolutionPool(solution_pool_size, alpha)

        self.model_dir = model_dir
        os.makedirs(self.model_dir, exist_ok=True)

        self.solution_dir = solution_dir
        self.hpwl_history = []

    def setup_tb_writer(self, tb_writer: SummaryWriter):
        self.tb_writer = tb_writer

    def train(self, agent: Agent, env: Environment):

        logging.info(f"==================== Training ====================")

        global_episode = 0
        benchmark_name = getattr(env, 'benchmark', 'unknown')
        for loop in range(self.num_loops):
            logging.info(
                f"-------------------- Loop {loop} --------------------")
            replay_buffer = ReplayBuffer(self.buffer_capacity, env.grid)
            for episode in range(self.num_episodes_in_loop):
                hpwl, sol_id, solution, placement = self.run_episode(
                    env, agent, replay_buffer, global_episode)

                logging.info(f"Episode {global_episode}, HPWL = {hpwl}")
                self.solution_pool.update_sol(
                    sol_id, solution, hpwl, placement)

                self.tb_writer.add_scalar(
                    'episodes/hpwl', hpwl, global_episode)
                self.tb_writer.add_scalar(
                    "episodes/best_hpwl", self.solution_pool.best_hpwl, global_episode)
                self.tb_writer.add_scalar(
                    "episodes/best_hpwl_max", self.solution_pool.best_hpwl_max, global_episode)
                self.tb_writer.add_scalar(
                    "episodes/depth_min", self.solution_pool.depth_min_max[0], global_episode)
                self.tb_writer.add_scalar(
                    "episodes/depth_max", self.solution_pool.depth_min_max[1], global_episode)
                self.hpwl_history.append({
                    'benchmark': benchmark_name,
                    'loop': loop,
                    'episode': episode,
                    'global_episode': global_episode,
                    'hpwl': float(hpwl),
                    'best_hpwl': float(self.solution_pool.best_hpwl),
                })
                global_episode += 1

            agent.update(replay_buffer, self.num_update_epochs, self.update_batch_size)

            if global_episode >= self.update_frontiers_begin and loop % self.update_frontiers_freq == 0:
                self.solution_pool.update_frontiers()

            placement, sol, hpwl = self.solution_pool.best_solution

            logging.info(f"Current best HPWL = {hpwl}.")
            torch.save(agent.actor_net.state_dict(),
                       os.path.join(self.model_dir, "actor_net.pth"))
            torch.save(agent.critic_net.state_dict(),
                       os.path.join(self.model_dir, "critic_net.pth"))

            os.makedirs(self.solution_dir, exist_ok=True)
            with open(os.path.join(self.solution_dir, f"best_placement_{hpwl}.pkl"), 'wb') as f:
                pickle.dump(placement, f)

        self.save_hpwl_history(benchmark_name)

    def save_hpwl_history(self, benchmark_name: str):
        csv_path = os.path.join(self.model_dir, "hpwl_history.csv")
        pkl_path = os.path.join(self.model_dir, "hpwl_history.pkl")
        with open(pkl_path, 'wb') as f:
            pickle.dump(self.hpwl_history, f)
        with open(csv_path, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=['benchmark', 'loop', 'episode', 'global_episode', 'hpwl', 'best_hpwl'])
            writer.writeheader()
            writer.writerows(self.hpwl_history)
        logging.info(f"Saved HPWL history for {benchmark_name} to {pkl_path}")

    def run_episode(self, env: Environment, agent: Agent, replay_buffer: ReplayBuffer, global_episode: int):
        # initialize the environment
        t, hpwl, score = 0, 0, 0
        solution, placement = [], []
        s = env.reset()
        done = False

        # randomly select a frontier to start an episode from
        sol_id, sol = self.solution_pool.sample()

        while True:
            if t < len(sol):
                # select the action from the solution
                a, a_logp = sol[t], 0.0
            else:
                a, a_logp = agent.select_action(s, t)

            s_, r, done, info = env.step(a)
            placement.append(a)
            hpwl += info["delta_hpwl"]

            if not done:
                if t >= len(sol):
                    replay_buffer.store(s, t, a, a_logp, r, s_, t + 1, 0.0)
                solution.append(a)
                score += r

                t += 1
            if done and env.num_macros > self.num_macros_to_place:
                # if the episode is done, place the remaining macros greedily
                s_done, s__done, a_done = s, s_, a
                s = s_
                for i in range(env.num_macros - self.num_macros_to_place):
                    a = agent.act_greedy(s)
                    s_, r_, done, info = env.step(a)
                    placement.append(a)
                    hpwl += info["delta_hpwl"]
                    r += r_
                    if i == env.num_macros - self.num_macros_to_place - 1:
                        replay_buffer.store(s_done, t, a_done, a_logp, r, s__done, t + 1, 1.0)
                        solution.append(a_done)
                    s = s_
                score += r
                break
                
            s = s_

        return hpwl * env.ratio / 1e5, sol_id, solution, placement


In [ ]:
def build_config(spec):
    run_root = OUTPUT_ROOT / spec['name'] / RUN_TAG
    benchmark_dir = benchmark_dir_for(spec['name'])
    return OmegaConf.create({
        'seed': 0,
        'cuda': 0,
        'benchmark': spec['name'],
        'tb_dir': str(run_root / 'tb'),
        'model_dir': str(run_root / 'model'),
        'solution_dir': str(run_root / 'sol'),
        'env': {
            'benchmark': spec['name'],
            'benchmark_dir': str(benchmark_dir),
            'rank_mode': spec['rank_mode'],
            'wire_mask_scale': spec['wire_mask_scale'],
            'reward_scale': spec['reward_scale'],
            'num_macros_to_place': spec['num_macros_to_place'],
            'grid': spec['grid'],
        },
        'agent': {
            'max_grad_norm': 0.5,
            'clip_epsilon': 0.2,
            'entropy_coef': 0.001,
            'lr_actor': spec['lr_actor'],
            'lr_critic': spec['lr_critic'],
            'actor_lr_anneal_rate': spec['actor_lr_anneal_rate'],
            'critic_lr_anneal_rate': spec['critic_lr_anneal_rate'],
            'num_macros_to_place': spec['num_macros_to_place'],
            'grid': spec['grid'],
            'lamda': 0.98,
            'gamma': spec['gamma'],
        },
        'trainer': {
            'num_loops': spec['num_loops'],
            'num_episodes_in_loop': spec['num_episodes_in_loop'],
            'num_update_epochs': spec['num_update_epochs'],
            'update_batch_size': spec['update_batch_size'],
            'num_macros_to_place': spec['num_macros_to_place'],
            'solution_pool_size': spec['solution_pool_size'],
            'alpha': spec['alpha'],
            'update_frontiers_begin': spec['update_frontiers_begin'],
            'update_frontiers_freq': spec['update_frontiers_freq'],
            'model_dir': str(run_root / 'model'),
            'solution_dir': str(run_root / 'sol'),
        },
    })


def run_one_benchmark(spec):
    config = build_config(spec)
    run_root = OUTPUT_ROOT / spec['name'] / RUN_TAG
    print('===== Running benchmark:', spec['name'], '=====')
    print(OmegaConf.to_yaml(config))
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    tb_writer = SummaryWriter(config.tb_dir)
    agent = Agent(**OmegaConf.to_container(config.agent, resolve=True))
    agent.set_up(device, tb_writer)
    env = Environment(**OmegaConf.to_container(config.env, resolve=True))
    trainer = Trainer(**OmegaConf.to_container(config.trainer, resolve=True))
    trainer.setup_tb_writer(tb_writer)
    trainer.train(agent, env)
    tb_writer.close()
    return {
        'benchmark': spec['name'],
        'run_root': str(run_root),
        'history_path': str(Path(config.model_dir) / 'hpwl_history.pkl'),
        'solution_dir': str(Path(config.solution_dir)),
    }


training_results = []
for spec in BENCHMARK_SPECS:
    training_results.append(run_one_benchmark(spec))

print('All benchmarks finished.')
print(training_results)

In [ ]:
# Instantiate the training components that are already defined inside the notebook.
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    torch.cuda.empty_cache()

print('Using device:', device)

tb_writer = SummaryWriter(config.tb_dir)
agent = Agent(**OmegaConf.to_container(config.agent, resolve=True))
agent.set_up(device, tb_writer)
env = Environment(**OmegaConf.to_container(config.env, resolve=True))
trainer = Trainer(**OmegaConf.to_container(config.trainer, resolve=True))
trainer.setup_tb_writer(tb_writer)

trainer.train(agent, env)
print('Training finished.')

## VISUAlIZE

In [ ]:
import os
import pickle
import matplotlib.pyplot as plt
import matplotlib.patches as patches


def load_hpwl_history(history_path):
    with open(history_path, 'rb') as f:
        history = pickle.load(f)
    episodes = [row['global_episode'] for row in history]
    hpwl = [row['hpwl'] for row in history]
    best_hpwl = [row['best_hpwl'] for row in history]
    return episodes, hpwl, best_hpwl


def plot_hpwl_convergence(benchmark_names=BENCHMARK_NAMES, run_tag=RUN_TAG):
    fig, ax = plt.subplots(figsize=(12, 6))
    plotted = False
    for benchmark_name in benchmark_names:
        history_path = OUTPUT_ROOT / benchmark_name / run_tag / 'model' / 'hpwl_history.pkl'
        if not history_path.exists():
            print(f'Missing history: {history_path}')
            continue
        episodes, hpwl, best_hpwl = load_hpwl_history(history_path)
        ax.plot(episodes, best_hpwl, label=benchmark_name, linewidth=1.5)
        plotted = True
    ax.set_xlabel('Global Episode')
    ax.set_ylabel('Best HPWL')
    ax.set_title('HPWL Convergence Across Benchmarks')
    ax.grid(True, alpha=0.3)
    if plotted:
        ax.legend(ncol=2, fontsize=8)
    plt.show()


def visualize_placement(pkl_path, env):
    with open(pkl_path, 'rb') as f:
        placement = pickle.load(f)

    fig, ax = plt.subplots(figsize=(10, 10))
    ax.set_xlim(0, env.grid)
    ax.set_ylim(0, env.grid)

    for t, action in enumerate(placement):
        pos_x = round(action // env.grid)
        pos_y = round(action % env.grid)
        size_x = env.size_x[t]
        size_y = env.size_y[t]
        rect = patches.Rectangle((pos_x, pos_y), size_x, size_y, linewidth=1, edgecolor='blue', facecolor='lightblue', alpha=0.6)
        ax.add_patch(rect)
        ax.text(pos_x + size_x / 2, pos_y + size_y / 2, str(t), fontsize=8, ha='center', va='center', color='black')

    filename = os.path.basename(pkl_path)
    hpwl_val = filename.replace('best_placement_', '').replace('.pkl', '')
    plt.title(f'Macro Placement: {filename}\n(HPWL: {hpwl_val})', fontsize=14)
    plt.xlabel('Grid X')
    plt.ylabel('Grid Y')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.show()


def visualize_latest_placement(benchmark_name, run_tag=RUN_TAG):
    spec = BENCHMARK_SPEC_MAP[benchmark_name]
    env = Environment(**{
        'benchmark': benchmark_name,
        'benchmark_dir': str(benchmark_dir_for(benchmark_name)),
        'rank_mode': spec['rank_mode'],
        'wire_mask_scale': spec['wire_mask_scale'],
        'reward_scale': spec['reward_scale'],
        'num_macros_to_place': spec['num_macros_to_place'],
        'grid': spec['grid'],
    })
    sol_dir = OUTPUT_ROOT / benchmark_name / run_tag / 'sol'
    pkl_files = sorted(sol_dir.glob('best_placement_*.pkl'))
    if not pkl_files:
        print(f'No placement file found for {benchmark_name} in {sol_dir}')
        return
    visualize_placement(pkl_files[-1], env)


plot_hpwl_convergence()
visualize_latest_placement(ACTIVE_BENCHMARK)